# Simple diagnostic of Bjerknes feedback

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time
import pandas as pd

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## funcs

In [ ]:
def make_scatter_ax(
    ax, anom_, xvar, yvar, month, label, verbose=True, by_season=True, **kwargs
):
    """scatter plot of data for given month"""

    ## prep func
    if by_season:
        get_season = lambda x: src.utils.sel_month(
            x.resample({"time": "QS-JAN"}).mean(), month
        )

    else:
        get_season = lambda x: src.utils.sel_month(x, month)

    prep = lambda x: get_season(x).transpose("time", "member")

    ## get plot data
    plot_data = (prep(anom_[xvar]), prep(anom_[yvar]))

    ## get stats
    corr = xr.corr(*plot_data)
    cov = xr.cov(*plot_data)
    m = cov / plot_data[0].var()

    ## get label
    if verbose:
        legend = f"m = {m.item():.1e}\nr = {corr.item():.2f}"
    else:
        legend = None

    ## plot data
    ax.scatter(*plot_data, s=3, label=legend, **kwargs)
    ax.set_title(f"{label}")

    ## formatting
    ax_kwargs = dict(ls="--", c="gray", lw=0.5)
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)

    if verbose:
        ax.legend(prop=dict(size=10))

    return ax


def make_scatter(anom_, xvar, yvar, month, by_season=True, verbose=True, **kwargs):
    """scatter plot of data for given month"""

    fig, axs = plt.subplots(1, 4, figsize=(11, 2.5), layout="constrained")

    if "year" in anom_:
        for ax, year in zip(
            axs,
            [1870, 2010, 2050, 2080],
        ):

            ## scatter plot of data
            ax = make_scatter_ax(
                ax,
                anom_.sel(year=year, method="nearest"),
                xvar=xvar,
                yvar=yvar,
                month=month,
                by_season=by_season,
                label=year,
                verbose=verbose,
                **kwargs,
            )

    else:

        for ax, t_idx, label in zip(
            axs,
            [["1850", "1879"], ["1995", "2024"], ["2035", "2064"], ["2071", "2100"]],
            ["1865", "2010", "2050", "2085"],
        ):

            ## helper func
            prep = lambda x: x.sel(time=slice(*t_idx))

            ## scatter plot of data
            ax = make_scatter_ax(
                ax,
                prep(anom_),
                xvar=xvar,
                yvar=yvar,
                month=month,
                by_season=by_season,
                label=label,
            )

    ## format/scale axes
    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_timeseries(coefs, sel_fn=lambda x: x):
    """plot timeseries comparison"""

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5))

    ## loop thru pos and negative
    for i, (name, color) in enumerate(zip(["pos", "neg"], ["r", "b"])):

        ## plot median and bounds
        for q, lw in zip([0.5, 0.1, 0.9], [2, 0.6, 0.6]):

            ## plot neutral and pos/or neg
            for name_, color_ in zip(["all", name], ["k", color]):

                ## finally, plot data
                axs[i].plot(
                    coefs.year,
                    sel_fn(coefs)[name_].quantile(q=q, dim="member"),
                    c=color_,
                    lw=lw,
                )

            ## plot on shared axs
            axs[2].plot(
                coefs.year,
                sel_fn(coefs)[name].quantile(q=q, dim="member"),
                c=color,
                lw=lw,
            )

    ## formatting
    for ax in axs:
        ax.set_xticks([1870, 2010, 2080])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axvline(2010, **ax_kwargs)
        ax.axhline(0, **ax_kwargs)
    src.utils.set_lims(axs)

    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_zonal_structure(coefs, sel_fn=lambda x: x):
    """plot zonal structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ax.plot(coefs.longitude, coefs_y, c="k")

    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    for ax in axs:
        ax.set_xticks([140, 190, 240, 280])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(0, **ax_kwargs)

    return fig, axs

## Load

In [ ]:
def prep(x, add_feats=True, vname="T_3"):
    """preprocess data"""

    ## get windowed version
    x_ = src.utils.get_windowed(x, stride=120).isel(year=slice(None,-1))

    ## subtract median
    median = x_.groupby("time.month").median(["time","member"])
    x_ = x_.groupby("time.month")-median

    ## add features
    if add_feats:
        x_ = add_features(x_, vname=vname)

    return x_

def prep_spatial(x, add_feats=True, vname="T_3"):
    """preprocess data"""

    ## get data
    x0 = x.sel(time=slice("1851", "1890"))
    x1 = x.sel(time=slice("1991", "2030"))
    x2 = x.sel(time=slice("2061", "2100"))

    ## make time coords match
    x1 = x1.assign_coords({"time":x0.time})
    x2 = x2.assign_coords({"time":x0.time})

    ## concat
    x_ = xr.concat([x0,x1,x2], dim=pd.Index([1870, 2010, 2080], name="year"))
    
    ## subtract median
    median = x_.groupby("time.month").median(["time","member"])
    x_ = x_.groupby("time.month")-median

    ## add features
    if add_feats:
        x_ = add_features(x_, vname=vname)

    return x_

def add_features(X, vname="T_3"):
    """add constant and quadratic term"""
    ones = xr.ones_like(X[vname]).rename("ones")
    quad = (X[vname]**2).rename("T2")
    return xr.merge([X, ones, quad])

def regress_pinv_spatial(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    # prep = lambda x: x.transpose("member", ...)
    prep = lambda x : x.stack(s=["time","member"]).transpose("year",...,"s")
    
    X_ = prep(X[x_vars].to_dataarray(dim="v"))
    Y_ = prep(X[y_var])

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year":X.year, "mode":X.mode, "v": x_vars},
        dims=["year","mode","v"],
    )

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("yki,yij->ykj", Y_.values, X_pinv)

    return m

def regress_pinv(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    # prep = lambda x: x.transpose("member", ...)
    prep = lambda x : x.stack(s=["time","member"]).transpose("year",...,"s")
    
    X_ = prep(X[x_vars].to_dataarray(dim="v"))
    Y_ = prep(X[y_var])

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year":X.year, "v": x_vars},
        dims=["year","v"],
    )

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("yi,yij->yj", Y_.values, X_pinv)

    return m

In [ ]:
## indices
Th = src.utils.load_cesm_indices(load_z20=False)

## spatial data
forced, anom = src.utils.load_consolidated()
anom = anom[["taux", "ssh", "T"]].assign_coords({"time":Th.time})

## merge and prep
X = xr.merge([Th, anom])

## Analysis

In [ ]:
def get_he(ssh):
    """get he"""
    idx = dict(longitude=slice(210, 280), latitude=slice(-5,5))
    
    return ssh.sel(idx).sum("longitude").mean("latitude")

def get_hw(ssh):
    """get hw"""

    idx = dict(longitude=slice(120, 170), latitude=slice(-5,5))
    
    return ssh.sel(idx).sum("longitude").mean("latitude")

def get_dh(ssh):
    return (get_he(ssh)-get_hw(ssh))/100

### sverdrup

compute

In [ ]:
## get data
beta_data = prep_spatial(X[["T_3", "ssh"]])

beta_T_proj = beta_data.groupby("time.season").map(
    regress_pinv_spatial, x_vars=["T_3", "T2", "ones"], y_var="ssh"
)

beta_T_struct = src.utils.reconstruct_fn(
    scores=beta_T_proj, 
    components=forced["ssh_comp"], 
    fn=lambda x : x.sel(latitude=slice(-2,2)).mean("latitude"),
)

Plot

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(9,3))

for ax, v in zip(axs, ["ones","T_3","T2"]):
    for y in [1870, 2010]:
        ax.plot(beta_T_struct.longitude, beta_T_struct.sel(year=y, v=v).mean("season"))
    ax.set_xlim([120,280])

    ax_kwargs = dict(ls="--", c="gray", lw=1)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(170, **ax_kwargs)

src.utils.set_lims(axs)

Get fits

In [ ]:
## compute dh
dh = src.utils.reconstruct_fn(
    components=forced["ssh_comp"],
    scores=X["ssh"],
    fn=get_dh,
)


## get data for regression
beta_data = prep(xr.merge([X["T_3"], dh.rename("dh")]))

In [ ]:
beta_vars = ["T_3", "T2", "ones", "dh"]
beta_T = beta_data.groupby("time.season").map(
    regress_pinv, x_vars=["T_3", "T2", "ones"], y_var="dh"
)

zz = np.linspace(-4,4)
ZZ = np.stack([zz**i for i in [1,2,0]])
ZZ_xr = xr.DataArray(
    ZZ, dims=["v","sigma"], coords=dict(v=beta_T.v, sigma=zz)
)

## get fits
fits = (beta_T * ZZ_xr).sum("v")

Scatter plot

In [ ]:
## SPECIFY season/month
STARTM = 9
SEASON = "SON"

Z = beta_data[["dh","T_3"]].resample({"time":"QS-DEC"}).mean()
Z = Z.sel(time=(Z.time.dt.month==STARTM))

fig,axs = plt.subplots(1,2,figsize=(6,3))

years = [1880, 2010]

for ax, y in zip(axs, years):
    
    ax.scatter(
        Z["T_3"].sel(year=y), Z["dh"].sel(year=y), s=3,
    )

    ax.plot(
        ZZ_xr.sigma, fits.sel(year=y, season=SEASON), c="k"
    )

    ax_kwargs = dict(ls="--", c="gray", lw=1)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)

axs[1].plot(
    ZZ_xr.sigma, fits.sel(year=years[0], season=SEASON), c="w", ls="--"
)
src.utils.set_lims(axs)
axs[1].set_yticks([])

plt.show()


## TIMESERIES
sel = lambda x : x.sel(season=SEASON).isel(year=slice(1,None))
beta_T_recon = fits.differentiate("sigma")


fig,ax = plt.subplots(figsize=(4,3))
for s_, c in zip([-1,0,1], ["b","k","r"]):
    ax.plot(
        beta_T.year.isel(year=slice(1,None)), 
        sel(beta_T_recon.sel(sigma=s_, method="nearest")), 
        c=c
    )
plt.show()

### thermal expansion

specify MLD

In [ ]:
MLD = 30

#### spatial

In [ ]:
## compute dh
he = src.utils.reconstruct_fn(
    components=forced["ssh_comp"],
    scores=X["ssh"],
    fn=get_he,
).rename("he")

## standardize
he = he/he.std()


## get data for regression
ah_data = prep_spatial(xr.merge([X[["T", "T_3"]], he]), vname="he")

In [ ]:
ah_proj = ah_data.groupby("time.season").map(
    regress_pinv_spatial, x_vars=["he", "T2", "ones"], y_var="T"
)

ah_struct = src.utils.reconstruct_fn(
    scores=ah_proj,
    components=forced["T_comp"],
    fn=lambda x : x.sel(z_t=slice(None,MLD)).mean("z_t"),
)

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(9,3))

for ax, v in zip(axs, ["ones","he","T2"]):
    for y in [1870, 2010]:
        ax.plot(ah_struct.longitude, ah_struct.sel(year=y, v=v).mean("season"))
    ax.set_xlim([120,280])

    ax_kwargs = dict(ls="--", c="gray", lw=1)
    ax.axhline(0, **ax_kwargs)

for ax in axs[1:]:
    ax.set_yticks([])

src.utils.add_vticks(axs, xticks=[210,270], xlines=[210,270])
src.utils.set_lims(axs)
plt.show()

#### index

In [ ]:
## compute Tsub
Tsub = src.utils.reconstruct_fn(
    components=forced["T_comp"],
    scores=X["T"],
    fn=lambda x : x.sel(
        z_t=slice(None,MLD), longitude=slice(210,270)
    ).mean(["longitude","z_t"]),
).rename("Tsub")


## get data for regression
ah_data = prep(xr.merge([Tsub, he]), vname="he")

In [ ]:
ah = ah_data.isel(year=slice(1,None)).groupby("time.season").map(
    regress_pinv, x_vars=["he", "T2", "ones"], y_var="Tsub"
)

zz = np.linspace(-3,3)
ZZ = np.stack([zz**i for i in [1,2,0]])
ZZ_xr = xr.DataArray(
    ZZ, dims=["v","sigma"], coords=dict(v=ah.v, sigma=zz)
)

## get fits
fits = (ah * ZZ_xr).sum("v")

In [ ]:
## SPECIFY season/month
STARTM = 12
SEASON = "DJF"

Z = ah_data[["he","Tsub"]].resample({"time":"QS-DEC"}).mean()
Z = Z.sel(time=(Z.time.dt.month==STARTM))

fig,axs = plt.subplots(1,2,figsize=(6,3))

years = [1880, 2080]

for ax, y, c, ls in zip(axs, years, ["k", "magenta"], ["-","--"]):
    
    ax.scatter(
        Z["he"].sel(year=y), Z["Tsub"].sel(year=y), s=3,
    )

    ax.plot(
        ZZ_xr.sigma, fits.sel(year=y, season=SEASON), c=c, ls=ls, zorder=10,
    )

    ax_kwargs = dict(ls="--", c="gray", lw=1)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)

axs[1].plot(
    ZZ_xr.sigma, fits.sel(year=years[0], season=SEASON), c="k", ls="-"
)
src.utils.set_lims(axs)
axs[1].set_yticks([])

plt.show()


## TIMESERIES
sel = lambda x : x.sel(season=SEASON).isel(year=slice(1,None))
# sel = lambda x : x.sel(season=SEASON).isel(year=slice(1,None))
ah_recon = fits.differentiate("sigma")

fig,ax = plt.subplots(figsize=(4,3))
for s_, c in zip([-1,0,1], ["b","k","r"]):
    ax.plot(
        ah.year.isel(year=slice(1,None)), 
        sel(ah_recon.sel(sigma=s_, method="nearest")), 
        c=c
    )

src.utils.add_vticks([ax], xticks=[2010], xlines=[2010])
plt.show()